# SER Training -- 6-Class Classification (mHuBERT-147)

Stage A: Extract mHuBERT-147 features -> cache. Stage B: Train classification head.

Prerequisite: ser_dataset_setup.ipynb

Output: MyDrive/CLEAR/emotion_data/feat_cache/ + CLEAR/models/


In [ ]:
# Cell 1: Drive + deps
import os, sys, csv, time, json
from pathlib import Path
from collections import Counter
import numpy as np
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/CLEAR/emotion_data")
AUDIO_DIR = DRIVE_ROOT / "audio"
LABELS_DIR = DRIVE_ROOT / "labels"
CACHE_DIR = DRIVE_ROOT / "feat_cache"
MODEL_DIR = DRIVE_ROOT.parent / "models"
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
print(f"Drive root: {DRIVE_ROOT}")
print(f"Cache: {CACHE_DIR}")
print(f"Models: {MODEL_DIR}")
!pip install -q librosa soundfile transformers datasets
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")


## Config


In [ ]:
# Cell 2: Config
CLASSES = ["panic", "fear", "urgency", "distress", "confusion", "neutral"]
N_CLASSES = 6
DATASETS = ["cremad", "kaggle_emergency", "ravdess", "jl", "asvp_esd"]
MAX_TRAIN_SAMPLES = None
TRAIN_BATCH = 256
EVAL_BATCH = 512
NUM_EPOCHS = 100
LR = 1e-3
LR_PATIENCE = 5
LR_FACTOR = 0.5
STOP_PATIENCE = 10
NUM_WORKERS = 2
BACKBONE_ID = "utter-project/mHuBERT-147"
SR = 16000
HID = 768
print("Config ready")


## Model + Dataset


In [ ]:
# Cell 3: EmotionHead
class EmotionHead(nn.Module):
    def __init__(self, hidden_size=HID, num_classes=N_CLASSES):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_size, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, num_classes),
        )
    def forward(self, x):
        return self.net(x.mean(dim=1))
print(f"EmotionHead: {sum(p.numel() for p in EmotionHead().parameters())} params")

# Cell 4: Dataset
class CachedFeatureDataset(Dataset):
    def __init__(self, csv_path):
        self.samples = []
        with open(csv_path) as f:
            for row in csv.DictReader(f):
                self.samples.append((row["feat_path"], int(row["class_idx"])))
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        feat = np.load(self.samples[idx][0]).astype(np.float32)
        return torch.from_numpy(feat), self.samples[idx][1]


## Stage A: Feature Extraction


In [ ]:
# Cell 5: Stage A
from transformers import HubertModel, Wav2Vec2FeatureExtractor
import librosa
import hashlib
t_start = time.time()
backbone = None
fe = None
for ds_name in DATASETS:
    csv_path = LABELS_DIR / f"{ds_name}_labels.csv"
    if not csv_path.exists():
        print(f"SKIP {ds_name}")
        continue
    with open(csv_path) as f: clips = list(csv.DictReader(f))
    if MAX_TRAIN_SAMPLES: clips = clips[:MAX_TRAIN_SAMPLES]
    feat_dir = CACHE_DIR / f"{ds_name}_feats"
    os.makedirs(feat_dir, exist_ok=True)
    if backbone is None:
        print("Loading mHuBERT-147...")
        backbone = HubertModel.from_pretrained(BACKBONE_ID)
        backbone.eval().to(DEVICE)
        fe = Wav2Vec2FeatureExtractor.from_pretrained(BACKBONE_ID)
        print(f"  Loaded")
    n_ok = 0
    print(f"[{ds_name}] {len(clips)} clips")
    for i, clip in enumerate(clips):
        key = hashlib.md5(clip["path"].encode()).hexdigest()[:12]
        fp = str(feat_dir / f"feat_{key}.npy")
        if os.path.exists(fp):
            n_ok += 1; continue
        try:
            audio, sr = librosa.load(clip["path"], sr=SR, mono=True)
        except: continue
        if len(audio) < 4000: continue
        if len(audio) > 8*SR: audio = audio[:8*SR]
        inputs = fe(audio, sampling_rate=SR, return_tensors="pt", padding=True, do_normalize=True).to(DEVICE)
        with torch.no_grad():
            feat = backbone(**inputs).last_hidden_state[0].cpu().numpy()
        np.save(fp, feat)
        n_ok += 1
        if (i+1) % 500 == 0: print(f"  [{i+1}/{len(clips)}] {time.time()-t_start:.0f}s")
    print(f"  OK={n_ok}")
print("Stage A complete.")


## Train/Val/Test Split


In [ ]:
# Cell 6: Split
import hashlib
from collections import defaultdict

print("Creating speaker-stratified split...")
class_to_idx = {c:i for i,c in enumerate(CLASSES)}
all_samples = []  # (feat_path, class_idx, dataset, speaker_id)

for ds_name in DATASETS:
    fd = CACHE_DIR / f"{ds_name}_feats"
    csv_path = LABELS_DIR / f"{ds_name}_labels.csv"
    if not csv_path.exists() or not fd.exists(): continue
    with open(csv_path) as f: clips = list(csv.DictReader(f))
    if MAX_TRAIN_SAMPLES: clips = clips[:MAX_TRAIN_SAMPLES]
    for i, clip in enumerate(clips):
        key = hashlib.md5(clip["path"].encode()).hexdigest()[:12]
        fp = fd / f"feat_{key}.npy"
        cn = clip.get("class_6", "")
        spk = clip.get("speaker_id", "unknown")
        if cn in class_to_idx and fp.exists():
            all_samples.append((str(fp), class_to_idx[cn], ds_name, spk))

print(f"Total samples: {len(all_samples)}")

# Build speaker groups
group_map = defaultdict(list)
for idx, (_, _, ds, spk) in enumerate(all_samples):
    group_map[(ds, spk)].append(idx)
groups = list(group_map.values())
rng = np.random.RandomState(42)
rng.shuffle(groups)

n = len(groups)
n_train = max(1, int(n * 0.8))
n_val = max(1, int(n * 0.1))
train_idx = [i for g in groups[:n_train] for i in g]
val_idx = [i for g in groups[n_train:n_train+n_val] for i in g]
test_idx = [i for g in groups[n_train+n_val:] for i in g]

print(f"Speaker groups: {len(groups)} (train={n_train} val={n_val} test={n-n_train-n_val})")
print(f"Samples: train={len(train_idx)} val={len(val_idx)} test={len(test_idx)}")

def write_split(path, indices):
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["feat_path", "class_idx", "dataset", "speaker_id"])
        for i in indices:
            fp, ci, ds, spk = all_samples[i]
            w.writerow([fp, ci, ds, spk])

write_split(CACHE_DIR / "train.csv", train_idx)
write_split(CACHE_DIR / "val.csv", val_idx)
write_split(CACHE_DIR / "test.csv", test_idx)

# Class distribution in train
train_cls = Counter(all_samples[i][1] for i in train_idx)
print("Train distribution:")
for i,c in enumerate(CLASSES): print(f"  {c}: {train_cls.get(i,0)}")
print(f"Distinct speaker groups in train: {len(set((all_samples[i][2], all_samples[i][3]) for i in train_idx))}")


## Stage B: Training


In [ ]:
# Cell 7: Training
from sklearn.metrics import classification_report
if "all_samples" not in dir() or not all_samples:
    print("ERROR: No samples. Run Cell 6.")
else:
    td = CachedFeatureDataset(CACHE_DIR/"train.csv")
    vd = CachedFeatureDataset(CACHE_DIR/"val.csv")
    tstd = CachedFeatureDataset(CACHE_DIR/"test.csv")
    tl = DataLoader(td, TRAIN_BATCH, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    vl = DataLoader(vd, EVAL_BATCH, num_workers=NUM_WORKERS, pin_memory=True)
    tsl = DataLoader(tstd, EVAL_BATCH, num_workers=NUM_WORKERS, pin_memory=True)
    model = EmotionHead(HID, N_CLASSES).to(DEVICE)
    train_cls = Counter(s[1] for s in all_samples[:int(len(all_samples)*0.8)])
    cnts = [train_cls.get(i,1) for i in range(N_CLASSES)]
    w = torch.tensor([sum(cnts)/max(c,1) for c in cnts], dtype=torch.float32).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=w)
    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", patience=LR_PATIENCE, factor=LR_FACTOR)
    best_loss = float("inf")
    best_ep = -1
    no_imp = 0
    for ep in range(1, NUM_EPOCHS+1):
        t0 = time.time()
        model.train()
        tr_l, tr_c, tr_n = 0,0,0
        for f,t in tl:
            f,t = f.to(DEVICE), t.to(DEVICE)
            l = criterion(model(f), t)
            opt.zero_grad(); l.backward(); opt.step()
            tr_l += l.item()*f.size(0)
            tr_c += (model(f).argmax(1)==t).sum().item()
            tr_n += f.size(0)
        model.eval()
        vl_l, vl_c, vl_n = 0,0,0
        with torch.no_grad():
            for f,t in vl:
                f,t = f.to(DEVICE), t.to(DEVICE)
                l = criterion(model(f), t)
                vl_l += l.item()*f.size(0)
                vl_c += (model(f).argmax(1)==t).sum().item()
                vl_n += f.size(0)
        sched.step(vl_l/vl_n)
        print(f"  Ep {ep:3d}: tr_l={tr_l/tr_n:.4f} acc={tr_c/tr_n:.4f} | val_l={vl_l/vl_n:.4f} acc={vl_c/vl_n:.4f} | {time.time()-t0:.1f}s")
        if vl_l/vl_n < best_loss:
            best_loss = vl_l/vl_n; best_ep = ep; no_imp = 0
            torch.save(model.state_dict(), MODEL_DIR/"best_emotion_head.pt")
        else:
            no_imp += 1
            if no_imp >= STOP_PATIENCE: break
    print(f"Best epoch: {best_ep}. Loading for test...")
    model.load_state_dict(torch.load(MODEL_DIR/"best_emotion_head.pt", map_location=DEVICE))
    model.eval()
    all_preds, all_targets = [], []
    conf = torch.zeros(N_CLASSES, N_CLASSES, dtype=torch.int64)
    with torch.no_grad():
        for f,t in tsl:
            f,t = f.to(DEVICE), t.to(DEVICE)
            p = model(f).argmax(1)
            for pi, ti in zip(p,t):
                conf[ti,pi] += 1
                all_preds.append(pi.item())
                all_targets.append(ti.item())

    print(f"\nConfusion matrix:")
    print("     " + "".join(f"{c:>6}" for c in CLASSES))
    for i,c in enumerate(CLASSES):
        row = "".join(f"{conf[i,j].item():>6}" for j in range(N_CLASSES))
        print(f"{c:>6}{row}")

    print(f"\nPer-class accuracy:")
    for i,c in enumerate(CLASSES):
        corr = conf[i,i].item(); tot = conf[i].sum().item()
        print(f"  {c}: {corr}/{tot} ({corr/max(tot,1):.3f})")

    print(f"\nClassification report:")
    print(classification_report(all_targets, all_preds, target_names=CLASSES, digits=3))
    torch.save(model.state_dict(), MODEL_DIR/"best_emotion_head.pt")
    print(f"Saved to {MODEL_DIR}/best_emotion_head.pt")
